# Vesti-Care — Train LSTM 3-class (Colab GPU)

**Trước khi chạy:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload 4 file lên Google Drive vào folder `VestiCare/data/`:
   - `ml/data/processed_3class/X_train.npz` (172 MB)
   - `ml/data/processed_3class/y_train.npz`
   - `ml/data/processed_3class/X_test.npz`
   - `ml/data/processed_3class/y_test.npz`
3. Chạy từng cell theo thứ tự

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — check runtime type!')

In [ ]:
# Cell 2 — Cau hinh duong dan
# Chinh DRIVE_DATA_DIR neu ban upload vao folder khac
DRIVE_DATA_DIR  = '/content/drive/MyDrive/VestiCare/data'
DRIVE_MODEL_DIR = '/content/drive/MyDrive/VestiCare/models'

import os
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# Kiem tra file ton tai
for f in ['X_train.npz', 'y_train.npz', 'X_test.npz', 'y_test.npz']:
    path = f'{DRIVE_DATA_DIR}/{f}'
    size = os.path.getsize(path) / 1024 / 1024 if os.path.exists(path) else -1
    status = f'{size:.1f} MB' if size >= 0 else 'NOT FOUND!'
    print(f'  {f}: {status}')

In [ ]:
# Cell 3 — Tham so (chinh o day)
FOCAL_GAMMA            = 2.0
FALL_WEIGHT_MULTIPLIER = 6.0
EPOCHS                 = 60
BATCH_SIZE             = 128   # GPU: tang batch size de nhanh hon
PATIENCE               = 12
LR_INIT                = 1e-3
LR_MIN                 = 1e-6
RECALL_TARGET          = 0.85

CLASS_NAMES  = ['NORMAL', 'INSTABILITY', 'FALL']
NUM_CLASSES  = 3
WINDOW_SIZE  = 100
NUM_CHANNELS = 7

print('Config OK')

In [ ]:
# Cell 4 — Import va load data
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

print(f'TF version: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

X_train = np.load(f'{DRIVE_DATA_DIR}/X_train.npz')['data'].astype(np.float32)
y_train = np.load(f'{DRIVE_DATA_DIR}/y_train.npz')['data'].astype(np.int32)
X_test  = np.load(f'{DRIVE_DATA_DIR}/X_test.npz')['data'].astype(np.float32)
y_test  = np.load(f'{DRIVE_DATA_DIR}/y_test.npz')['data'].astype(np.int32)

print(f'\nTrain: {X_train.shape}  Test: {X_test.shape}')
for i, n in enumerate(CLASS_NAMES):
    print(f'  {n:25s}: train={int((y_train==i).sum()):6d}  test={int((y_test==i).sum()):5d}')

In [ ]:
# Cell 5 — Augment + Normalize
def augment_minority_classes(X, y, seed=42):
    rng = np.random.default_rng(seed)
    normal_count = np.sum(y == 0)
    aug_X, aug_y = [X], [y]
    for cls in [1, 2]:
        mask  = (y == cls)
        X_cls = X[mask]
        count = len(X_cls)
        if count == 0:
            continue
        repeats = max(1, (normal_count // count) - 1)
        print(f'  Class {CLASS_NAMES[cls]}: {count} -> x{repeats+1}')
        for _ in range(repeats):
            X_new  = X_cls.copy()
            choice = rng.integers(0, 3, size=len(X_new))
            jmask = (choice == 0)
            if jmask.any():
                X_new[jmask] += rng.normal(0, 0.05, X_new[jmask].shape)
            smask = (choice == 1)
            if smask.any():
                scale = rng.uniform(0.85, 1.15, (smask.sum(), 1, 1))
                X_new[smask] *= scale
            tmask = (choice == 2)
            if tmask.any():
                shift = int(rng.integers(-3, 4))
                X_new[tmask] = np.roll(X_new[tmask], shift, axis=1)
            aug_X.append(X_new)
            aug_y.append(np.full(len(X_new), cls, dtype=np.int32))
    X_aug = np.concatenate(aug_X)
    y_aug = np.concatenate(aug_y)
    perm  = rng.permutation(len(X_aug))
    return X_aug[perm], y_aug[perm]

print('Augmenting...')
X_train_aug, y_train_aug = augment_minority_classes(X_train, y_train)
print(f'After augment: {len(X_train_aug)} windows')

mean = X_train_aug.mean(axis=(0, 1))
std  = X_train_aug.std(axis=(0, 1))
std[std == 0] = 1.0
X_tr_norm = (X_train_aug - mean) / std
X_te_norm  = (X_test      - mean) / std
print('Normalization done')

In [ ]:
# Cell 6 — Focal loss + class weights
def focal_loss(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        p_t     = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal_w = (1.0 - p_t) ** gamma
        ce      = -tf.math.log(p_t)
        return tf.reduce_mean(focal_w * ce)
    loss_fn.__name__ = f'focal_loss_g{gamma}'
    return loss_fn

cw_vals = compute_class_weight('balanced',
                               classes=np.arange(NUM_CLASSES),
                               y=y_train_aug)
cw_vals[2] *= FALL_WEIGHT_MULTIPLIER
class_weight = {i: float(w) for i, w in enumerate(cw_vals)}
print(f'Class weights: {class_weight}')

y_tr_cat = tf.keras.utils.to_categorical(y_train_aug, NUM_CLASSES)
y_te_cat = tf.keras.utils.to_categorical(y_test,      NUM_CLASSES)

In [ ]:
# Cell 7 — Kien truc LSTM
from tensorflow.keras import layers, models, regularizers

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))
x   = layers.LSTM(64, return_sequences=True,
                  kernel_regularizer=regularizers.l2(1e-4))(inp)
x   = layers.Dropout(0.3)(x)
x   = layers.LSTM(32, return_sequences=False,
                  kernel_regularizer=regularizers.l2(1e-4))(x)
x   = layers.Dropout(0.2)(x)
x   = layers.Dense(32, activation='relu')(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=inp, outputs=out, name='lstm_3class')
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_INIT),
    loss=focal_loss(FOCAL_GAMMA),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# Cell 8 — Train
import time

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=LR_MIN, verbose=1,
    ),
]

t0 = time.time()
history = model.fit(
    X_tr_norm, y_tr_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_te_norm, y_te_cat),
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)
print(f'\nTraining done in {(time.time()-t0)/60:.1f} minutes')

In [ ]:
# Cell 9 — Evaluate
from sklearn.metrics import confusion_matrix, classification_report

probs  = model.predict(X_te_norm, batch_size=512, verbose=0)
y_pred = np.zeros(len(y_test), dtype=np.int32)
y_pred[probs[:, 1] >= 0.50] = 1
y_pred[probs[:, 2] >= 0.45] = 2

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
print('Confusion Matrix:')
print(f'  {"":12s}  PRED_NRM  PRED_INST  PRED_FALL')
for i, name in enumerate(CLASS_NAMES):
    print(f'  TRUE_{name:10s}  {cm[i,0]:8d}  {cm[i,1]:9d}  {cm[i,2]:9d}')

fall_tp = float(cm[2, 2])
fall_fn = float(cm[2, 0] + cm[2, 1])
fall_fp = float(cm[0, 2] + cm[1, 2])
recall    = fall_tp / (fall_tp + fall_fn) if (fall_tp + fall_fn) > 0 else 0
precision = fall_tp / (fall_tp + fall_fp) if (fall_tp + fall_fp) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
inst_r    = float(cm[1,1]) / float(cm[1].sum()) if cm[1].sum() > 0 else 0

print(f'\n  FALL  recall={recall:.4f}  prec={precision:.4f}  F1={f1:.4f}')
print(f'  INST  recall={inst_r:.4f}')
status = 'PASS' if recall >= RECALL_TARGET else 'BELOW_TARGET'
print(f'\n  [{status}] FALL recall={recall:.4f}  (target={RECALL_TARGET})')

In [ ]:
# Cell 10 — Luu model vao Drive + download ve may
import os

# ---- 1. Luu Keras ----
keras_path = f'{DRIVE_MODEL_DIR}/fall_detection_3class_lstm.keras'
model.save(keras_path)
size_kb = os.path.getsize(keras_path) / 1024
print(f'[OK] Keras saved: {keras_path}  ({size_kb:.0f} KB)')

# ---- 2. Convert TFLite (LSTM can SELECT_TF_OPS) ----
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    converter._experimental_lower_tensor_list_ops = False
    tflite_bytes = converter.convert()
    tflite_path  = f'{DRIVE_MODEL_DIR}/fall_detection_3class_lstm.tflite'
    with open(tflite_path, 'wb') as f:
        f.write(tflite_bytes)
    print(f'[OK] TFLite saved: {tflite_path}  ({len(tflite_bytes)/1024:.0f} KB)')
    tflite_ok = True
except Exception as e:
    print(f'[WARN] TFLite convert failed: {e}')
    print('       Chi can file .keras la du chay baseline_comparison.py')
    tflite_ok = False

# ---- 3. Download ve may ----
from google.colab import files
print('\nDownloading files...')
files.download(keras_path)
if tflite_ok:
    files.download(tflite_path)
print('\n[DONE] Copy file vao ml/models_3class/ roi chay:')
print('       python ml/scripts/baseline_comparison.py')